# Service Alert Delay Correlation

Compares observed delays during active service alerts with observations without active alerts.

In [1]:
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "analysis"))

spec = importlib.util.spec_from_file_location(
    "service_alert_delay_correlation",
    PROJECT_ROOT / "analysis" / "service-alert-delay-correlation.py",
)
service_alerts = importlib.util.module_from_spec(spec)
spec.loader.exec_module(service_alerts)

DB = PROJECT_ROOT / "data" / "foli.db"
GTFS_DIR = None
GTFS_ROOT = PROJECT_ROOT / "data" / "gtfs"
LIMIT = 20
MIN_OBSERVATIONS = 100
ALERT_KIND = "route"  # "route", "stop", or "any"
LINE_REF = None
START = None
END = None
ANALYSIS_DAYS = 2
FULL_HISTORY = False
TIMEZONE = "Europe/Helsinki"
QUALITY_MODE = "conservative"
BUCKET = "trip-stop"

Alert rows are grouped by cause, effect, priority, and route/stop scope. Controls are matched from non-alert buckets with the same line, direction, local hour, and weekday/weekend context. By default this notebook analyzes the latest two days of observations to avoid loading the full database into pandas; set `START` and `END` for a custom window.

In [2]:
class Args:
    db = DB
    gtfs_dir = GTFS_DIR
    gtfs_root = GTFS_ROOT
    limit = LIMIT
    min_observations = MIN_OBSERVATIONS
    alert_kind = ALERT_KIND
    line_ref = LINE_REF
    start = START
    end = END
    analysis_days = ANALYSIS_DAYS
    full_history = FULL_HISTORY
    timezone = TIMEZONE
    quality_mode = QUALITY_MODE
    exclude_stop_call_disagreement = False
    bucket = BUCKET

grouped, line = service_alerts.build_correlation(Args)
grouped

""


In [3]:
line

""


In [4]:
if not line.empty:
    ax = line.sort_values("p90_delay_lift_min").plot.barh(
        x="line_ref",
        y="p90_delay_lift_min",
        legend=False,
        figsize=(8, 6),
        title="Delay lift during active service alerts by line",
    )
    ax.set_xlabel("Alert minus no-alert P90 delay, minutes")
    ax.set_ylabel("Line")